In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [44]:
import random


def get_weather(city: str) -> str:
    """
    Use this tool when the user asks about the current weather, 
    temperature, or climate conditions of a specific city.
    
    Args:
        city: The name of the city (e.g., 'London', 'New York').
    """
    temp = random.randint(15, 35)
    return f"The weather in {city} is sunny now. temperature is {temp} degree."

In [54]:
def transportation_info(city: str) -> str:
    """ 
     Use this tool when the user asks about the transportation like bus,
     taxi, metro rail etc of a specific city

     Args:
        city: The name of the city (e.g., 'London', 'New York').
    """
    return f"All the public buses in {city} are unavailable today. Metro rail and taxi are available"

In [3]:
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    temperature=0.1,
    task="text-generation",
    max_new_tokens=512,
)

chat_model = ChatHuggingFace(llm = llm)

e:\Personal Project\my_first_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(
    groq_api_key = groq_api_key,
    model="llama-3.1-8b-instant",
    temperature=0.0,
    max_retries=2,
    max_tokens=1024
)

system_instruction = (
    "You are a helpful assistant. Use your tools to gather real-time data.\n\n"
    "CRITICAL REQUIREMENT:\n"
    "When tools return information, you MUST directly answer the user's question "
    "using that specific data. Combine the findings from all tools into a short, "
    "Do not just output a generic disclaimer."
)

agent = create_agent(
    model=llm,
    tools=[get_weather,transportation_info],
    system_prompt=system_instruction
)

In [81]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "how is transportations situation in Mumbai now and what is the temperature today?"}]},
    config={"recursion_limit": 5} # Cap it at 5 graph transitions max
)
response

{'messages': [HumanMessage(content='how is transportations situation in Mumbai now and what is the temperature today?', additional_kwargs={}, response_metadata={}, id='13a3730a-c1f7-4f68-94fc-3e879c67b2b6'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'hxmyh5s62', 'function': {'arguments': '{"city":"Mumbai"}', 'name': 'transportation_info'}, 'type': 'function'}, {'id': 'eprw139a4', 'function': {'arguments': '{"city":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 121, 'prompt_tokens': 496, 'total_tokens': 617, 'completion_time': 0.228499724, 'completion_tokens_details': None, 'prompt_time': 0.050018754, 'prompt_tokens_details': None, 'queue_time': 0.056185565, 'total_time': 0.278518478}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f64d0-b93b-7c23-bb

In [6]:
response = llm.invoke("Who won last fifa world cup?")
print(response.content)

The 2022 FIFA World Cup was won by Argentina, with Lionel Messi being named the tournament's best player. They defeated France 4-2 in a penalty shootout after the match ended 3-3 after extra time in the final on December 18, 2022.


RAG for tool calling in agent

In [143]:
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.agents import create_agent
from langchain_huggingface import HuggingFaceEmbeddings 
from dotenv import load_dotenv

load_dotenv()

# 1. Define a pool of specialized tools
@tool
def get_crypto_price(ticker: str) -> str:
    """Get the live price of a cryptocurrency ticker (e.g., BTC, ETH)."""
    return f"The price of {ticker} is $65,000."

@tool
def get_weather(city: str) -> str:
    """Get the current weather forecast for a given city."""
    return f"The weather in {city} is sunny and 72°F."

@tool
def calculate_mortgage(principal: int, rate: float, years: int) -> str:
    """Calculate monthly mortgage payments based on loan details."""
    # Dummy calculation for demonstration
    return f"Your monthly payment for a ${principal} loan is $2,100."

# Keep all available tools in a dictionary mapping their name -> tool object
tool_pool = {
    "get_crypto_price": get_crypto_price,
    "get_weather": get_weather,
    "calculate_mortgage": calculate_mortgage
}

# 2. Build the Tool Index (RAG for Tools)
# We store the tool descriptions in a vector store to search them semantically
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
tool_vector_store = InMemoryVectorStore(embeddings)

# Add tools to our vector store using their descriptions as the index text
for name, tool_obj in tool_pool.items():
    tool_vector_store.add_texts(
        texts=[tool_obj.description],
        metadatas=[{"name": name}]
    )

# 3. Dynamic Retrieval Function
def retrieve_relevant_tools(user_query: str, k: int = 5) -> list:
    """Finds the most contextually appropriate tools for the query."""
    # Search the vector store for descriptions matching the user's intent
    docs = tool_vector_store.similarity_search(user_query, k=k)

    print(docs)
    
    # Map the matched metadata back to the actual tool objects
    selected_tools = [tool_pool[doc.metadata["name"]] for doc in docs]
    return selected_tools



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2207.58it/s]


In [144]:
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model="llama-3.3-70b-versatile",
    temperature=0.0,
    max_retries=2,
    max_tokens=1024
)

In [ ]:
def run_agent_with_retrieved_tools(user_query: str):
    # Tightened the instruction to remove the Llama disclaimer loophole
    system_instruction = (
    "You are a helpful assistant with access to tools for real-time data.\n\n"
    "WORKFLOW (follow exactly):\n"
    "1. Call a tool at most ONCE per user question.\n"
    "2. As soon as a tool returns a result, STOP calling tools and immediately "
    "write your final answer in plain text using the specific data returned "
    "(numbers, conditions, etc.) — never a generic disclaimer.\n"
    "3. Only call a tool a second time if the first call returned an explicit error.\n"
    "4. Do not call any tool after you have already produced a final answer."
    )
    
    # Dynamic retrieval
    retrieved_tools = retrieve_relevant_tools(user_query, k=2)
    
    print(f"\n🔮 User Query: '{user_query}'")
    print(f"🗂️ Retaining only the tool: {[t.name for t in retrieved_tools]}")
    
    # Initialize agent
    agent = create_agent(model=llm, tools=retrieved_tools, system_prompt=system_instruction)
    
    # response = agent.invoke({"messages": [{"role": "user", "content": user_query}]},
    # config={"recursion_limit": 10})
    # final_answer = response["messages"][-1].content
    # print(f"🤖 Agent Answer:\n{final_answer}")

    for step in agent.stream(
    {"messages": [{"role": "user", "content": user_query}]},
    config={"recursion_limit": 10},
    stream_mode="values",):
        last = step["messages"][-1]
        print(type(last).__name__, "->", getattr(last, "tool_calls", None) or last.content)
    

In [155]:
# --- Test Runs ---
run_agent_with_retrieved_tools("How is the weather in Dhaka today And how much is Bitcoin worth right now?")
# run_agent_with_retrieved_tools("How much is Bitcoin worth right now?")

[Document(id='d389136b-27a6-4b97-9071-ca47ed245ac6', metadata={'name': 'get_crypto_price'}, page_content='Get the live price of a cryptocurrency ticker (e.g., BTC, ETH).'), Document(id='c6544dc3-e27f-4552-876f-dd3a00be7f90', metadata={'name': 'get_weather'}, page_content='Get the current weather forecast for a given city.')]

🔮 User Query: 'How is the weather in Dhaka today And how much is Bitcoin worth right now?'
🗂️ Retaining only the tool: ['get_crypto_price', 'get_weather']
HumanMessage -> How is the weather in Dhaka today And how much is Bitcoin worth right now?
AIMessage -> [{'name': 'get_weather', 'args': {'city': 'Dhaka'}, 'id': 't9yap7rvh', 'type': 'tool_call'}, {'name': 'get_crypto_price', 'args': {'ticker': 'BTC'}, 'id': '7bemnveb4', 'type': 'tool_call'}]
ToolMessage -> The price of BTC is $65,000.
AIMessage -> The weather in Dhaka is sunny and 72°F. The price of BTC is $65,000.
